In [18]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_google_genai import GoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv

from langchain.retrievers.multi_query import MultiQueryRetriever

In [6]:
load_dotenv()

True

In [7]:
# Health and wellness documents

docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}, id = '1'),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}, id = '2'),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}, id = '3'),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}, id = '4'),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}, id = '5'),
    
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}, id = '6'),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}, id = '7'),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}, id = '8'),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}, id = '9'),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}, id = '10'),
]

In [24]:
model_emb = OpenAIEmbeddings(
    model = 'text-embedding-3-large',
    dimensions = 32
)

model_gemini = GoogleGenerativeAI(
    model = 'gemini-2.5-flash'
)

vec_store = FAISS.from_documents(
    documents = docs,
    embedding = model_emb
)

In [25]:
#Now, we will creater two retriever objects, 
#one simple_retriever and other multi_query_retriever ans see the dfference


# Simple retriever
simple_retriever = vec_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 3}
)

# Milti query retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever = simple_retriever,    # We can also use 'mmr'
    llm = model_gemini
)

In [26]:
# Defining our query

simple_result = simple_retriever.invoke("How to imporve energy and maintain balance?")
multi_result = multi_query_retriever.invoke("How to imporve energy and maintain balance?")

In [27]:
simple_result

[Document(id='8', metadata={'source': 'I3'}, page_content='Photosynthesis enables plants to produce energy by converting sunlight.'),
 Document(id='2', metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'),
 Document(id='3', metadata={'source': 'H3'}, page_content='Deep sleep is crucial for cellular repair and emotional regulation.')]

In [28]:
multi_result

# This might return some irrelevant documents


# This is because words 'energy' and 'balance' confused our retrievers
# and hence they also gave these irrelevant documents

[Document(id='1', metadata={'source': 'H1'}, page_content='Regular walking boosts heart health and can reduce symptoms of depression.'),
 Document(id='8', metadata={'source': 'I3'}, page_content='Photosynthesis enables plants to produce energy by converting sunlight.'),
 Document(id='2', metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'),
 Document(id='3', metadata={'source': 'H3'}, page_content='Deep sleep is crucial for cellular repair and emotional regulation.'),
 Document(id='10', metadata={'source': 'I5'}, page_content='Black holes bend spacetime and store immense gravitational energy.')]

# Now go to word file